In [424]:
#Import libraries
#Use pandas for data tables
#Use requests to connect to Transfermarkt
#Use StringIO to read HTML content safely

import pandas as pd
import requests
from io import StringIO

In [425]:
#Create request headers
#This helps the script act like a normal browser request

headers = {
    "User-Agent": "Mozilla/5.0"
}

In [426]:
#Define Transfermarkt attendance URLs
#Page 1 has the first group of MLS clubs
#Page 2 has the remaining MLS clubs

urls = [
    "https://www.transfermarkt.us/major-league-soccer/besucherzahlen/wettbewerb/MLS1",
    "https://www.transfermarkt.us/major-league-soccer/besucherzahlen/wettbewerb/MLS1/page/2"
]

In [427]:
#Create an empty list
#Store the attendance table from each page

all_tables = []

In [428]:
#Loop through each URL
#Download each page
#Extract the attendance table
#Remove empty rows
#Store each table

for url in urls:
    response = requests.get(url, headers=headers)
    tables = pd.read_html(StringIO(response.text))
    
    table = tables[1].copy()
    table = table.dropna(subset=["Average"])
    
    all_tables.append(table)

In [429]:
#Combine both attendance tables
#Create one complete dataframe with all MLS clubs from page 1 and page 2

df = pd.concat(all_tables, ignore_index=True)

In [430]:
#Remove Total rows
#Total rows are summaries, not teams

df = df[~df["Stadium"].astype(str).str.contains("Total", na=False)]
df = df.reset_index(drop=True)

In [431]:
#Create complete MLS team list
#Use exact Transfermarkt names

teams = [
    "Atlanta United FC",
    "Austin FC",
    "CF Montréal",
    "Charlotte FC",
    "Chicago Fire FC",
    "Colorado Rapids",
    "Columbus Crew",
    "D.C. United",
    "FC Cincinnati",
    "FC Dallas",
    "Houston Dynamo FC",
    "Inter Miami CF",
    "Los Angeles FC",
    "Los Angeles Galaxy",
    "Minnesota United FC",
    "Nashville SC",
    "New England Revolution",
    "New York City FC",
    "Red Bull New York",
    "Orlando City SC",
    "Philadelphia Union",
    "Portland Timbers",
    "Real Salt Lake City",
    "San Diego FC",
    "San Jose Earthquakes",
    "Seattle Sounders FC",
    "Sporting Kansas City",
    "St. Louis CITY SC",
    "Toronto FC",
    "Vancouver Whitecaps FC"
]

In [432]:
#Extract team names from the Stadium column
#Each row contains stadium name + team name together

def extract_team(text):
    for team in teams:
        if team in text:
            return team
    return None

df["Team"] = df["Stadium"].apply(extract_team)

In [433]:
#Validate team extraction
#Confirm that all 30 MLS teams were detected

print("Total Teams:", df["Team"].nunique())
df[df["Team"].isna()][["Stadium"]]

Total Teams: 30


,Stadium


In [434]:
#Clean numeric columns
#Transfermarkt uses dots as thousands separators
#Example: 20.738 means 20,738

numeric_columns = ["Capacity", "Spectators", "Average"]

for col in numeric_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(".", "", regex=False)
        .astype(int)
    )

In [435]:
#Fix capacity values that lost zeros
#Values below 1000 should be multiplied by 100

df["Capacity"] = df["Capacity"].apply(
    lambda x: x * 100 if x < 1000 else x
)

In [436]:
#Fix remaining capacity values manually
#Some stadium capacities still lost one zero during conversion

capacity_corrections = {
    "Colorado Rapids": 33680,
    "Seattle Sounders FC": 68740
}

for team, capacity in capacity_corrections.items():
    df.loc[df["Team"] == team, "Capacity"] = capacity

In [437]:
#Fix average attendance values that lost zeros
#Some values were imported as 2078 instead of 20780

average_attendance_corrections = {
    "Real Salt Lake City": 20780,
    "New England Revolution": 17930,
    "Sporting Kansas City": 16630
}

for team, average in average_attendance_corrections.items():
    df.loc[df["Team"] == team, "Average"] = average

In [438]:
#Fix LAFC and D.C. United average attendance
#These values were too high compared to stadium capacity

average_attendance_corrections_2 = {
    "Los Angeles FC": 21714,
    "D.C. United": 20821
}

for team, average in average_attendance_corrections_2.items():
    df.loc[df["Team"] == team, "Average"] = average

In [439]:
#Create Capacity Utilization metric
#Average Attendance divided by Stadium Capacity times 100

df["Capacity Utilization"] = (
    df["Average"] / df["Capacity"]
) * 100

df["Capacity Utilization"] = df["Capacity Utilization"].round(2)

In [440]:
#Validate capacity utilization
#Values slightly above 100 can happen, but extreme values indicate errors

df[df["Capacity Utilization"] > 110]


,#,Stadium,Capacity,Spectators,Average,Team,Capacity Utilization


In [441]:
#Rename columns for clarity

df = df.rename(columns={
    "Spectators": "Total Spectators",
    "Average": "Average Attendance"
})

In [442]:
#Add Instagram follower counts manually
#Follower counts measure digital audience reach

instagram_followers = {
    "Atlanta United FC": 570000,
    "Austin FC": 241000,
    "CF Montréal": 292000,
    "Charlotte FC": 281000,
    "Chicago Fire FC": 263000,
    "Colorado Rapids": 142000,
    "Columbus Crew": 271000,
    "D.C. United": 230000,
    "FC Cincinnati": 205000,
    "FC Dallas": 450000,
    "Houston Dynamo FC": 239000,
    "Inter Miami CF": 18000000,
    "Los Angeles FC": 1300000,
    "Los Angeles Galaxy": 1500000,
    "Minnesota United FC": 243000,
    "Nashville SC": 194000,
    "New England Revolution": 170000,
    "New York City FC": 767000,
    "Red Bull New York": 297000,
    "Orlando City SC": 501000,
    "Philadelphia Union": 554000,
    "Portland Timbers": 207000,
    "Real Salt Lake City": 157000,
    "San Diego FC": 252000,
    "San Jose Earthquakes": 196000,
    "Seattle Sounders FC": 410000,
    "Sporting Kansas City": 218000,
    "St. Louis CITY SC": 179000,
    "Toronto FC": 366000,
    "Vancouver Whitecaps FC": 368000
}

In [443]:
#Map Instagram followers to each team
#Create digital reach column

df["Instagram Followers"] = df["Team"].map(instagram_followers)

In [444]:
#Validate Instagram follower mapping
#Check if any club is missing follower data

df[df["Instagram Followers"].isna()][["Team"]]

,Team


In [445]:
#Create Reach Efficiency metric
#Instagram Followers divided by Average Attendance

df["Reach Efficiency"] = (
    df["Instagram Followers"] / df["Average Attendance"]
).round(2)

In [446]:
#Select final Metric 1 columns
#Create clean dataset for Tableau

df_final = df[[
    "Team",
    "Capacity",
    "Total Spectators",
    "Average Attendance",
    "Capacity Utilization",
    "Instagram Followers",
    "Reach Efficiency"
]]

In [447]:
#Preview final dataset
#Verify values before exporting

df_final

,Team,Capacity,Total Spectators,Average Attendance,Capacity Utilization,Instagram Followers,Reach Efficiency
0,Atlanta United FC,73019,301014,37627,51.53,570000,15.15
1,Los Angeles FC,22000,207995,21714,98.70,1300000,59.87
2,Charlotte FC,75412,199589,28513,37.81,281000,9.86
3,Vancouver Whitecaps FC,54500,193515,24189,44.38,368000,15.21
4,New York City FC,30321,179643,22455,74.06,767000,34.16
5,Toronto FC,30991,173977,19331,62.38,366000,18.93
6,San Diego FC,35000,164965,23566,67.33,252000,10.69
7,Seattle Sounders FC,68740,154241,30848,44.88,410000,13.29
8,FC Cincinnati,26000,153078,25513,98.13,205000,8.04
9,Real Salt Lake City,20213,145108,20780,102.81,157000,7.56


In [448]:
df_final.to_csv(
    "../data/processed/mls_metric1_final.csv",
    index=False
)